# Visualización de rutas EVRP con matplotlib (sin folium)

Versión estática del notebook 02: solo usa `pandas`, `numpy` y `matplotlib`, sin mapas web.

Muestra para cada instancia:

1. Las rutas de cada vehículo en coordenadas lat/lon (un color por vehículo), con BASE, visitas y cargadores.
2. El perfil de SOC de cada vehículo, con la línea del SOC mínimo.
3. Una tabla resumen por vehículo.

Requisito: haber generado las salidas antes, desde la raíz del proyecto:

```bash
python -m src.solve --instance all --vehicles 2
```

Ejecuta con **Restart Kernel + Run All**, usando el kernel del `.venv` del proyecto.

In [ ]:
%matplotlib inline
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print('Python:', sys.executable)
print('matplotlib', matplotlib.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)


def find_project_root() -> Path:
    # Busca hacia arriba una carpeta que tenga 'data' y 'outputs'.
    # Así funciona aunque el kernel arranque en otra carpeta.
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data').is_dir() and (p / 'outputs').is_dir():
            return p
    raise FileNotFoundError(
        'No encuentro las carpetas data/ y outputs/. Abre el notebook dentro del proyecto '
        'y corre antes: python -m src.solve --instance all --vehicles 2'
    )


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / 'data'
OUTPUTS = PROJECT_ROOT / 'outputs'
FIGURES = OUTPUTS / 'figures'
SAVE_FIGURES = True  # guarda PNG en outputs/figures/ además de mostrarlos
print('Proyecto:', PROJECT_ROOT)

In [ ]:
VEHICLE_COLORS = list(plt.cm.tab10.colors)


def vehicle_color(vid):
    return VEHICLE_COLORS[int(vid) % len(VEHICLE_COLORS)]


def load_outputs(instance):
    route_path = OUTPUTS / f'route_{instance}.csv'
    metrics_path = OUTPUTS / f'metrics_{instance}.json'
    if not route_path.exists() or not metrics_path.exists():
        raise FileNotFoundError(
            f'Faltan salidas de {instance!r}. Corre: python -m src.solve --instance {instance} --vehicles 2'
        )
    route = pd.read_csv(route_path)
    metrics = json.loads(metrics_path.read_text())

    # Compatibilidad con salidas antiguas de 1 vehículo (sin columna 'vehicle').
    if 'vehicle' not in route.columns:
        route['vehicle'] = 0
    for col in ['soc_arrival_kwh', 'arc_energy_kwh', 'arc_km', 'recharged_kwh', 'recharge_time_h']:
        if col in route.columns:
            route[col] = pd.to_numeric(route[col], errors='coerce').fillna(0.0)
    route = route.sort_values(['vehicle', 'order']).reset_index(drop=True)

    nodes = pd.read_csv(DATA / f'{instance}_routing_nodes.csv')
    return route, metrics, nodes


def battery_params(route, metrics):
    # Lee capacidad y SOC mínimo del JSON, con valores de respaldo.
    cap = float(metrics.get('battery_capacity_kwh', route['soc_arrival_kwh'].max()))
    min_frac = float(metrics.get('min_soc_limit_frac', 0.10))
    return cap, min_frac

In [ ]:
def plot_routes(instance, route, metrics, nodes, label_nodes=True):
    fig, ax = plt.subplots(figsize=(10, 9))

    # Fondo: todos los cargadores de la instancia (gris).
    ch = nodes[nodes['node_type'] == 'charger']
    ax.scatter(ch['longitude'], ch['latitude'], marker='^', s=22, c='lightgray',
               edgecolors='gray', linewidths=0.5, zorder=1, label='Cargador (no usado)')

    used_charger_label = 'Cargador usado'
    for vid, g in route.groupby('vehicle'):
        color = vehicle_color(vid)
        xs = g['longitude'].to_numpy()
        ys = g['latitude'].to_numpy()
        n_vis = int((g['node_type'] == 'visit').sum())
        e_kwh = float(g['arc_energy_kwh'].sum())
        ax.plot(xs, ys, '-', color=color, lw=2, alpha=0.85, zorder=2,
                label=f'Vehículo {vid}: {n_vis} visitas, {e_kwh:.1f} kWh')

        # Flechas de dirección en la mitad de cada tramo.
        for i in range(len(xs) - 1):
            mx, my = (xs[i] + xs[i + 1]) / 2, (ys[i] + ys[i + 1]) / 2
            ax.annotate('', xy=(mx, my), xytext=(xs[i], ys[i]),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.3), zorder=3)

        vis = g[g['node_type'] == 'visit']
        ax.scatter(vis['longitude'], vis['latitude'], s=55, color=color,
                   edgecolors='black', linewidths=0.6, zorder=4)
        if label_nodes:
            for _, r in vis.iterrows():
                ax.annotate(r['node_id'], (r['longitude'], r['latitude']),
                            xytext=(4, 4), textcoords='offset points', fontsize=8, zorder=6)

        used = g[g['node_type'] == 'charger']
        if len(used):
            ax.scatter(used['longitude'], used['latitude'], marker='^', s=160, c='limegreen',
                       edgecolors='black', linewidths=1, zorder=5, label=used_charger_label)
            used_charger_label = None  # una sola entrada en la leyenda
            for _, r in used.iterrows():
                ax.annotate(f"{r['node_id']} (+{r['recharged_kwh']:.1f} kWh)",
                            (r['longitude'], r['latitude']), xytext=(6, -12),
                            textcoords='offset points', fontsize=8, color='darkgreen', zorder=6)

    base = route[route['node_type'] == 'base'].iloc[0]
    ax.scatter([base['longitude']], [base['latitude']], marker='*', s=400, c='black',
               zorder=7, label='BASE')

    # Encuadre: extensión de las rutas + margen (los cargadores lejanos quedan fuera).
    pad_x = (route['longitude'].max() - route['longitude'].min()) * 0.08 + 1e-3
    pad_y = (route['latitude'].max() - route['latitude'].min()) * 0.08 + 1e-3
    ax.set_xlim(route['longitude'].min() - pad_x, route['longitude'].max() + pad_x)
    ax.set_ylim(route['latitude'].min() - pad_y, route['latitude'].max() + pad_y)
    # Corrige la deformación lat/lon para que las distancias se vean proporcionales.
    ax.set_aspect(1.0 / np.cos(np.deg2rad(route['latitude'].mean())))

    total_e = metrics.get('total_drive_energy_kwh', metrics.get('drive_energy_kwh', route['arc_energy_kwh'].sum()))
    fleet = metrics.get('fleet_size', route['vehicle'].nunique())
    ax.set_title(f'Rutas - {instance} | flota {fleet} | energía total {total_e:.2f} kWh')
    ax.set_xlabel('Longitud')
    ax.set_ylabel('Latitud')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)
    fig.tight_layout()
    return fig

In [ ]:
def plot_soc(instance, route, metrics):
    cap, min_frac = battery_params(route, metrics)
    min_kwh = cap * min_frac
    vehicles = sorted(route['vehicle'].unique())
    fig, axes = plt.subplots(len(vehicles), 1, figsize=(12, 3.4 * len(vehicles)),
                             squeeze=False, sharey=True)

    for ax, vid in zip(axes[:, 0], vehicles):
        g = route[route['vehicle'] == vid]
        color = vehicle_color(vid)
        x = np.arange(len(g))
        soc = g['soc_arrival_kwh'].to_numpy()

        ax.axhspan(0, min_kwh, color='red', alpha=0.08)
        ax.axhline(min_kwh, color='red', ls='--', lw=1, label=f'SOC mínimo ({min_frac:.0%})')
        ax.axhline(cap, color='green', ls=':', lw=1, label=f'Capacidad ({cap:.1f} kWh)')
        ax.plot(x, soc, marker='o', color=color, lw=2, label='SOC al llegar')

        is_ch = (g['node_type'] == 'charger').to_numpy()
        if is_ch.any():
            ax.scatter(x[is_ch], soc[is_ch], marker='^', s=140, c='limegreen',
                       edgecolors='black', zorder=5, label='Parada de recarga')

        ax.set_xticks(x)
        ax.set_xticklabels(g['node_id'], rotation=90, fontsize=8)
        ax.set_ylim(0, cap * 1.05)
        ax.set_ylabel('SOC [kWh]')
        ax.set_title(f'Vehículo {vid}: SOC mínimo alcanzado {soc.min() / cap:.0%}', fontsize=10)
        ax.grid(alpha=0.3)
        ax.legend(loc='lower left', fontsize=8)

    fig.suptitle(f'Perfil de SOC por vehículo - {instance}', fontsize=12)
    fig.tight_layout()
    return fig


def summary_table(route, metrics):
    cap, _ = battery_params(route, metrics)
    rows = []
    for vid, g in route.groupby('vehicle'):
        rows.append({
            'vehiculo': vid,
            'visitas': int((g['node_type'] == 'visit').sum()),
            'energia_kwh': round(float(g['arc_energy_kwh'].sum()), 2),
            'soc_min_%': round(100 * float(g['soc_arrival_kwh'].min()) / cap, 1),
            'recargas': int((g['recharged_kwh'] > 0).sum()),
            'energia_recargada_kwh': round(float(g['recharged_kwh'].sum()), 2),
            'tiempo_recarga_min': round(60 * float(g['recharge_time_h'].sum()), 1),
            'ruta': ' -> '.join(g['node_id']),
        })
    return pd.DataFrame(rows)


def show_instance(instance):
    route, metrics, nodes = load_outputs(instance)
    print(f"Instancia: {instance} | factible: {metrics.get('feasible')} | "
          f"visitas: {metrics.get('n_visits_covered')}/{metrics.get('n_visits_required')}")
    fig_r = plot_routes(instance, route, metrics, nodes)
    fig_s = plot_soc(instance, route, metrics)
    if SAVE_FIGURES:
        FIGURES.mkdir(parents=True, exist_ok=True)
        fig_r.savefig(FIGURES / f'{instance}_rutas.png', dpi=150, bbox_inches='tight')
        fig_s.savefig(FIGURES / f'{instance}_soc.png', dpi=150, bbox_inches='tight')
        print('Figuras guardadas en', FIGURES)
    plt.show()
    return summary_table(route, metrics)

## Instancia: evaluation

In [ ]:
show_instance('evaluation')

## Instancia: competition

In [ ]:
show_instance('competition')

## Ruta por calles sobre el mapa

Por defecto el solver optimiza con distancias por la red vial de OpenStreetMap (`distance_mode = road`: camino más corto dirigido, respetando calles de un sentido). Aquí se toma el **orden de paradas** que eligió y se reconstruye, tramo a tramo, ese mismo camino por calles para dibujarlo sobre un mapa base.

- Es el camino más corto por longitud, no una traza GPS.
- La tabla compara los km que usó el solver (`km_solver`, columna `arc_km` del CSV) con los km reconstruidos por calles. En modo `road` deben coincidir (`factor_calles` ≈ 1.00); si las salidas se generaron con `--distance haversine`, el factor muestra cuánto subestima la línea recta.
- El fondo del mapa son las propias calles de la red vial, dibujadas en gris desde el grafo local (`outputs/cache/`). No usa mosaicos externos, así que no necesita internet ni API key (el servidor público de OpenStreetMap bloquea las descargas automáticas y CARTO exige clave).

In [ ]:
from matplotlib.collections import LineCollection
from pyproj import Transformer

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import street_routes as sr

TO_MERCATOR = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)


def to_xy(lats, lons):
    x, y = TO_MERCATOR.transform(np.asarray(lons), np.asarray(lats))
    return np.asarray(x), np.asarray(y)


def kwh_per_km():
    params = OUTPUTS / 'models' / 'consumption_params.json'
    return float(json.loads(params.read_text())['kwh_per_km'])


print('Cargando red vial (la primera vez se descarga de OpenStreetMap)...')
DRIVE_GRAPH = sr.load_drive_graph()
print(f'Red vial: {len(DRIVE_GRAPH.nodes)} intersecciones, {len(DRIVE_GRAPH.edges)} tramos de calle')

# Fondo del mapa: todas las calles de la red en coordenadas Web Mercator.
# Se dibujan en gris claro bajo las rutas, sin mosaicos externos (el servidor
# de OpenStreetMap bloquea las descargas automáticas y CARTO exige API key).
_STREET_SEGMENTS_XY = [
    list(zip(*to_xy([lat for lat, lon in seg], [lon for lat, lon in seg])))
    for seg in sr.graph_edge_segments(DRIVE_GRAPH)
]


def draw_street_background(ax):
    """Dibuja la red vial local en gris claro como fondo del gráfico."""
    lc = LineCollection(_STREET_SEGMENTS_XY, colors='0.8', linewidths=0.6, zorder=0)
    ax.add_collection(lc)
    ax.set_facecolor('0.97')


def plot_streets(instance):
    route, metrics, _ = load_outputs(instance)
    cap, min_frac = battery_params(route, metrics)
    k = kwh_per_km()
    street = sr.street_routes_for_instance(route, DRIVE_GRAPH)

    fig, ax = plt.subplots(figsize=(12, 11))
    draw_street_background(ax)  # red vial local (gris) como fondo
    rows = []
    for sroute in street:
        vid = sroute.vehicle
        color = vehicle_color(vid)
        g = route[route['vehicle'] == vid].reset_index(drop=True)
        lats, lons = zip(*sroute.coords)
        x, y = to_xy(lats, lons)
        road_kwh = sroute.road_km * k
        ax.plot(x, y, '-', color=color, lw=3, alpha=0.85, zorder=3,
                label=f'Vehículo {vid}: {sroute.road_km:.1f} km por calles')

        # Paradas numeradas en el orden en que se visitan.
        sx, sy = to_xy(g['latitude'], g['longitude'])
        for i, r in g.iterrows():
            if r['node_type'] == 'visit':
                ax.scatter(sx[i], sy[i], s=60, color=color, edgecolors='black', zorder=5)
                ax.annotate(f"{i}. {r['node_id']}", (sx[i], sy[i]), xytext=(5, 5),
                            textcoords='offset points', fontsize=8, zorder=6,
                            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))
            elif r['node_type'] == 'charger':
                ax.scatter(sx[i], sy[i], marker='^', s=180, c='limegreen', edgecolors='black', zorder=6)
                ax.annotate(r['node_id'], (sx[i], sy[i]), xytext=(6, -12),
                            textcoords='offset points', fontsize=8, color='darkgreen', zorder=6)

        # km del solver: columna arc_km (salidas antiguas: energía / kWh por km).
        solver_km = float(g['arc_km'].sum()) if 'arc_km' in g.columns else float(g['arc_energy_kwh'].sum()) / k
        recharged = float(g['recharged_kwh'].sum())
        rows.append({
            'vehiculo': vid,
            'distancias_solver': metrics.get('distance_mode', 'haversine'),
            'km_solver': round(solver_km, 2),
            'km_por_calles': round(sroute.road_km, 2),
            'factor_calles': round(sroute.road_km / solver_km, 2) if solver_km > 0 else None,
            'energia_solver_kwh': round(float(g['arc_energy_kwh'].sum()), 2),
            'energia_por_calles_kwh': round(road_kwh, 2),
            'soc_final_por_calles_%': round(100 * (cap - road_kwh + recharged) / cap, 1),
        })

    bx, by = to_xy([route.iloc[0]['latitude']], [route.iloc[0]['longitude']])
    ax.scatter(bx, by, marker='*', s=500, c='black', zorder=7, label='BASE')

    # Encuadre con margen y proporción real (metros).
    all_lat = [c[0] for s in street for c in s.coords]
    all_lon = [c[1] for s in street for c in s.coords]
    ax_x, ax_y = to_xy(all_lat, all_lon)
    pad = 0.05 * max(ax_x.max() - ax_x.min(), ax_y.max() - ax_y.min())
    ax.set_xlim(ax_x.min() - pad, ax_x.max() + pad)
    ax.set_ylim(ax_y.min() - pad, ax_y.max() + pad)
    ax.set_aspect('equal')
    ax.set_axis_off()
    total_km = sum(s.road_km for s in street)
    ax.set_title(f'Ruta por calles - {instance} | {total_km:.1f} km | '
                 f'energía estimada {total_km * k:.1f} kWh', fontsize=13)
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURES.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURES / f'{instance}_calles.png', dpi=150, bbox_inches='tight')
    plt.show()
    return pd.DataFrame(rows)

In [ ]:
plot_streets('evaluation')

In [ ]:
plot_streets('competition')